In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier

print("Loading data...")
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

X_train_full = train_df.drop(columns=['id', 'class'])
y_train = train_df['class']
X_test_full = test_df.drop(columns=['id'])

# Feature Engineering: Adding "Color" differences to boost astronomical data accuracy
for df in [X_train_full, X_test_full]:
    df['u_g'] = df['u'] - df['g']
    df['g_r'] = df['g'] - df['r']
    df['r_i'] = df['r'] - df['i']
    df['i_z'] = df['i'] - df['z']

numeric_features = ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'u_g', 'g_r', 'r_i', 'i_z']
categorical_features = ['spectral_type', 'galaxy_population']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
    ]
)

print("Preprocessing...")
X_train_preprocessed = preprocessor.fit_transform(X_train_full)
X_test_preprocessed = preprocessor.transform(X_test_full)

# Fast parameter grid (only 4 combinations, 3 folds = 12 total fits)
hgb = HistGradientBoostingClassifier(random_state=42)
param_grid = {
    'learning_rate': [0.05, 0.1],
    'max_iter': [100, 150]
}

print("Starting GridSearch...")
grid_search = GridSearchCV(estimator=hgb, param_grid=param_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=2)
grid_search.fit(X_train_preprocessed, y_train)

print(f"Best Score: {grid_search.best_score_:.4f}")

# Predict directly without proba
best_model = grid_search.best_estimator_
predictions = best_model.predict(X_test_preprocessed)

submission = pd.DataFrame({'id': test_df['id'], 'class': predictions})
submission.to_csv('submission.csv', index=False)
print("Saved submission.csv")

Loading data...
Preprocessing...
Starting GridSearch...
Fitting 3 folds for each of 4 candidates, totalling 12 fits
Best Score: 0.9634
Saved submission.csv
